# Anomaly contextualization (_Cardio_)

In [2]:
from experiments.utils.constants import RANDOM_SEED, SOM_LEARNING_RATE_DECAY_FN, SOM_FIT_METHOD

VERBOSE = True
DATASET_NAME = "Cardio"
DATASET_ID = 193

MODELING_MODE = False
EXPORT_MODE = False
EXPORT_DIR = "_exports"

print(DATASET_NAME)

Cardio


## Dataset

In [3]:
# fetch dataset
from ucimlrepo import fetch_ucirepo
dataset = fetch_ucirepo(id=DATASET_ID)
X = dataset.data.features.values
y = dataset.data.targets["NSP"].values.ravel()
print(f"Dataset shape: {X.shape}, {y.shape}")

Dataset shape: (2126, 21), (2126,)


In [4]:
# prepare dataset
def split_data(X, y, verbose=True):
    import numpy as np
    from sklearn.model_selection import train_test_split

    X_hist, X_holdout, y_hist, y_holdout =  train_test_split(X, y, test_size=0.5, random_state=RANDOM_SEED, shuffle=True, stratify=y)

    if verbose:
        print(f"Historical set: {X_hist.shape} / {y_hist.shape}")
        print("\t > The data points grouped by label:")
        print(np.unique(y_hist, return_counts=True))

        print(f"\nHoldout set: {X_holdout.shape} / {y_holdout.shape}")
        print("\t > The data points grouped by label:")
        print(np.unique(y_holdout, return_counts=True))

    X_hist_normal = X_hist[y_hist == 1]
    X_hist_suspect = X_hist[y_hist == 2]
    X_hist_abnormal = X_hist[y_hist == 3]

    X_holdout_normal = X_holdout[y_holdout == 1]
    X_holdout_suspect = X_holdout[y_holdout == 2]
    X_holdout_abnormal = X_holdout[y_holdout == 3]

    return (X_hist, X_holdout, y_hist, y_holdout),  (X_hist_normal, X_hist_suspect, X_hist_abnormal, X_holdout_normal, X_holdout_suspect, X_holdout_abnormal)

In [5]:
(X_hist, X_holdout, y_hist, y_holdout), (X_hist_normal, X_hist_suspect, X_hist_abnormal, X_holdout_normal, X_holdout_suspect, X_holdout_abnormal) = split_data(X, y)
del X

Historical set: (1063, 21) / (1063,)
	 > The data points grouped by label:
(array([1, 2, 3]), array([827, 148,  88]))

Holdout set: (1063, 21) / (1063,)
	 > The data points grouped by label:
(array([1, 2, 3]), array([828, 147,  88]))


In [6]:
# scale data
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
Xs_hist_normal = scaler.fit_transform(X_hist_normal)
del X_hist_normal

## Modeling

In [7]:
from minisom_representation import calc_som_hyparams, SomRepresentation, plot_som_convergence_over_epochs

In [8]:
# use helper methods to get SOM hyperparameter recommendations
recommended_params = calc_som_hyparams(Xs_hist_normal, initial_sigma_factor=3.0)
print("Recommended SOM parameters:", recommended_params)

Recommended SOM parameters: {'d1': 12, 'd2': 13, 'sigma': 4.33}


In [9]:
# define actual hyperparameters
d1, d2, sigma = map(recommended_params.get, ("d1", "d2", "sigma"))
decay_function = SOM_LEARNING_RATE_DECAY_FN
epoch = None

In [10]:
# test candidate values for `num_iteration` hyperparameter
if MODELING_MODE:
    fig, errors_qe, errors_te = plot_som_convergence_over_epochs(
        SomRepresentation(d1=d1, d2=d2, sigma=sigma, random_seed=RANDOM_SEED, verbose=False, decay_function=decay_function),
        Xs_hist_normal,
        fit_type=SOM_FIT_METHOD, te_ceiling=.1,
        epoch_step_from=2, epoch_step_to=50, epoch_step=1,
        figsize=(16, 5), show_fig=True, verbose=VERBOSE
    )
    print(f"\nQE (first -> last): \t {errors_qe[0]:.2f} -> {errors_qe[-1]:.2f}")
    print(f"TE (first -> last): \t {errors_te[0]:.2f} -> {errors_te[-1]:.2f}")
else: print("Skipping model candidate parameters evaluation.")

Skipping model candidate parameters evaluation.


In [11]:
# set selected `num_iteration` as epoch
epoch = 42

In [12]:
# fit SOM representation
som_rep = SomRepresentation(d1=d1, d2=d2, sigma=sigma, random_seed=RANDOM_SEED, verbose=VERBOSE, decay_function=decay_function) \
    .fit_online(Xs_hist_normal, num_iteration=epoch)

 [ 34734 / 34734 ] 100% - 0:00:00 left 
 quantization error: 2.563632346269022

 An SOM representation has been fitted as follows:
------------------------------------------------------- 

Fit strategy: online 

Hyperparameters of SOM: 

{'input_len': 21, 'x': 12, 'y': 13, 'sigma': 4.33, 'topology': 'rectangular', 'learning_rate': 0.5, 'decay_function': 'linear_decay_to_zero', 'sigma_decay_function': 'asymptotic_decay', 'neighborhood_function': 'gaussian', 'activation_distance': 'euclidean', 'random_seed': 42, 'num_iteration': 42, 'use_epochs': True, 'random_order': True, 'verbose': True} 

Quality of SOM: 

Quantization Error (QE):	2.563632346269022
Topographic Error (TE): 	0.0036275695284159614


### Transform holdout data

In [13]:
Xs_hist_suspect = scaler.transform(X_hist_suspect)
Xs_hist_abnormal = scaler.transform(X_hist_abnormal)

Xs_holdout_suspect = scaler.transform(X_holdout_suspect)
Xs_holdout_abnormal = scaler.transform(X_holdout_abnormal)

del X_hist_suspect, X_hist_abnormal, X_holdout_suspect, X_holdout_abnormal

## Inspection

In [14]:
from utils.plotting import PlotlyHelperArgs

In [15]:
# create Basin
from lilypond import Basin
basin = Basin.from_som_representation(som_rep, random_seed=RANDOM_SEED, verbose=VERBOSE)

In [16]:
# export basin
if EXPORT_MODE:
    from utils.export import BasinWithTrainingData
    BasinWithTrainingData(dataset_name=DATASET_NAME, basin=basin, X_train=Xs_hist_normal).export("_exports/other")

In [17]:
# lilypond visual
basin.pond(base_style="iceflock") \
    .rhizome_layer() \
    .pad_layer() \
    .petal_layer() \
    .visualize(width=800, height=600);

## Extra figures

In [18]:
plot_args_base = dict(
    **PlotlyHelperArgs.Figsize(w=470, h=450),
    **PlotlyHelperArgs.FullStretch,
    font=dict(size=35),
    showlegend=False
)

plot_args = dict(
    **plot_args_base,
    **PlotlyHelperArgs.HiddenTicks(d1=d1, d2=d2),
)

uniform_black_colorscale = ["#000000", "#000000"]
rhizome_layer_style = dict(colorscale=uniform_black_colorscale, min_width=7, max_width=20)
marker_size = 25
marker_opacity = .9
marker_jitter_amount = .25

In [19]:
fig1 = basin.pond(base_style="iceflock") \
    .pad_layer(gap="nogap") \
    .rhizome_layer(**rhizome_layer_style) \
    .visualize(**plot_args_base);

In [20]:
fig2 = basin.pond(base_style="iceflock") \
    .pad_layer(gap="nogap") \
    .rhizome_layer(X=Xs_hist_suspect, **rhizome_layer_style) \
    .attraction_layer(Xs_hist_suspect, jitter_amount=marker_jitter_amount, marker=dict(color="tomato", symbol="star-diamond", size=marker_size, opacity=marker_opacity)) \
    .visualize(**plot_args);

In [21]:
fig3 = basin.pond(base_style="iceflock") \
    .pad_layer(gap="nogap") \
    .rhizome_layer(X=Xs_hist_abnormal, **rhizome_layer_style) \
    .attraction_layer(Xs_hist_abnormal, jitter_amount=marker_jitter_amount, marker=dict(color="orangered", symbol="star-diamond", size=marker_size, opacity=marker_opacity)) \
    .visualize(**plot_args);

In [22]:
fig4 = basin.pond(base_style="iceflock") \
    .pad_layer(gap="nogap") \
    .rhizome_layer(X=Xs_holdout_abnormal, **rhizome_layer_style) \
    .attraction_layer(Xs_holdout_abnormal, jitter_amount=marker_jitter_amount, marker=dict(color="magenta", symbol="star-diamond", size=marker_size, opacity=marker_opacity)) \
    .visualize(**plot_args);

In [23]:
if EXPORT_MODE:
    fig1.write_image(EXPORT_DIR + "/04_lilypond_01.png")
    fig2.write_image(EXPORT_DIR + "/04_lilypond_02.png")
    fig3.write_image(EXPORT_DIR + "/04_lilypond_03.png")
    fig4.write_image(EXPORT_DIR + "/04_lilypond_04.png")

---

### The below cells are not part of the experiment. They are used to persist the data and register the model in Databricks and Bianor for further interactive investigation.

---

## Preparation

In [25]:
MODEL_REGISTRATION_MODE = True
DATASET_PERSIST_MODE = True

In [26]:
import pandas as pd
import mlflow

from dotenv import load_dotenv
from utils.databricks_util import get_spark, get_catalog_path, CATALOG, SCHEMA

In [27]:
load_dotenv()
spark = get_spark()

## Persist data in Databricks

In [36]:
TABLE_NAME = f"T_{DATASET_NAME}".lower()
TABLE_HIST_PATH = get_catalog_path(TABLE_NAME + "_hist")
TABLE_HOLDOUT_PATH = get_catalog_path(TABLE_NAME + "_holdout")
print(TABLE_HIST_PATH)
print(TABLE_HOLDOUT_PATH)

workspace.lilypond_experiments.t_cardio_hist
workspace.lilypond_experiments.t_cardio_holdout


In [37]:
if DATASET_PERSIST_MODE:
	# persist full datasets as a managed table

    spark.createDataFrame(
		pd.DataFrame(X_hist) \
			.assign(label=y_hist) \
			.reset_index(names="id") \
        	.rename(columns={"label": "class"})
	).write \
		.mode("overwrite") \
		.saveAsTable(TABLE_HIST_PATH)

    spark.createDataFrame(
		pd.DataFrame(X_holdout) \
			.assign(label=y_holdout) \
			.reset_index(names="id") \
            .assign(id=lambda df: df["id"] + len(X_hist)) \
        	.rename(columns={"label": "class"})
	).write \
		.mode("overwrite") \
		.saveAsTable(TABLE_HOLDOUT_PATH)

In [38]:
data_dbdf_hist = spark.read \
    .table(TABLE_HIST_PATH)
data_dbdf_hist.show(5)

+---+-----+-----+-----+-----+-----+---+---+----+---+----+----+-----+-----+-----+----+---+-----+-----+-----+----+---+-----+
| id|    0|    1|    2|    3|    4|  5|  6|   7|  8|   9|  10|   11|   12|   13|  14| 15|   16|   17|   18|  19| 20|class|
+---+-----+-----+-----+-----+-----+---+---+----+---+----+----+-----+-----+-----+----+---+-----+-----+-----+----+---+-----+
|  0|144.0|0.001|  0.0|0.005|  0.0|0.0|0.0|30.0|1.1| 0.0|13.5| 43.0|122.0|165.0| 0.0|0.0|148.0|147.0|149.0| 3.0|0.0|    1|
|  1|117.0|  0.0|0.011|  0.0|  0.0|0.0|0.0|51.0|0.8| 9.0|13.3| 77.0| 56.0|133.0| 5.0|0.0|123.0|122.0|124.0| 2.0|1.0|    1|
|  2|122.0|0.002|0.003|0.006|0.002|0.0|0.0|20.0|5.0| 0.0|21.1|148.0| 50.0|198.0|11.0|0.0|127.0|124.0|127.0|28.0|0.0|    1|
|  3|127.0|0.009|  0.0|0.007|0.001|0.0|0.0|53.0|2.7| 0.0| 4.3|111.0| 71.0|182.0| 6.0|1.0|139.0|124.0|139.0|15.0|0.0|    1|
|  4|149.0|  0.0|  0.0|0.009|0.009|0.0|0.0|39.0|2.5|25.0| 5.6|137.0| 51.0|188.0| 4.0|1.0|155.0|139.0|153.0|97.0|1.0|    1|
+---+-----+-----

In [39]:
data_dbdf_holdout = spark.read \
    .table(TABLE_HOLDOUT_PATH)
data_dbdf_holdout.show(5)

+----+-----+-----+-----+-----+-----+---+---+----+---+----+----+----+-----+-----+---+---+-----+-----+-----+----+----+-----+
|  id|    0|    1|    2|    3|    4|  5|  6|   7|  8|   9|  10|  11|   12|   13| 14| 15|   16|   17|   18|  19|  20|class|
+----+-----+-----+-----+-----+-----+---+---+----+---+----+----+----+-----+-----+---+---+-----+-----+-----+----+----+-----+
|1063|121.0|0.006|  0.0|0.009|  0.0|0.0|0.0|39.0|0.9| 0.0| 8.6|58.0| 95.0|153.0|4.0|0.0|150.0|131.0|132.0|47.0| 0.0|    1|
|1064|121.0|0.004|0.002|0.002|0.005|0.0|0.0|56.0|1.6| 0.0|13.6|99.0| 70.0|169.0|5.0|1.0|124.0|121.0|124.0|28.0| 0.0|    1|
|1065|142.0|  0.0|  0.0|0.008|  0.0|0.0|0.0|58.0|0.4|22.0| 6.3|13.0|145.0|158.0|0.0|0.0|153.0|151.0|153.0| 0.0| 0.0|    1|
|1066|126.0|0.002|  0.0|0.009|  0.0|0.0|0.0|26.0|1.3| 0.0| 9.0|51.0|114.0|165.0|1.0|0.0|129.0|132.0|131.0|10.0|-1.0|    1|
|1067|124.0|0.008|  0.0|0.004|  0.0|0.0|0.0|46.0|1.0| 0.0| 9.3|92.0| 63.0|155.0|6.0|0.0|133.0|133.0|134.0| 7.0| 1.0|    1|
+----+-----+----

In [40]:
# separate variables
primary_key = ['id']
target = ['class']
data_df = pd.concat([data_dbdf_hist.toPandas(), data_dbdf_holdout.toPandas()], ignore_index=True)
features = data_df.columns.difference((primary_key + target), sort=False).tolist()
assert primary_key[0] not in features, "Primary key shall not be in Features"
assert all(t not in features for t in target), "Targets must not be in Features"
print("Primary key:", primary_key)
print("Targets:", target)
print("Features:", features)

Primary key: ['id']
Targets: ['class']
Features: ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20']


In [44]:
# create feature dataframe
data_dbdf = spark.createDataFrame(data_df)
feature_dbdf = data_dbdf.select(primary_key + features)
feature_dbdf.show(5)

+---+-----+-----+-----+-----+---+---+---+----+---+----+----+-----+-----+-----+---+---+-----+-----+-----+----+----+
| id|    0|    1|    2|    3|  4|  5|  6|   7|  8|   9|  10|   11|   12|   13| 14| 15|   16|   17|   18|  19|  20|
+---+-----+-----+-----+-----+---+---+---+----+---+----+----+-----+-----+-----+---+---+-----+-----+-----+----+----+
|531|147.0|  0.0|  0.0|  0.0|0.0|0.0|0.0|63.0|0.4|12.0| 8.1| 80.0| 74.0|154.0|1.0|0.0|148.0|147.0|149.0| 1.0| 1.0|
|532|115.0|0.008|  0.0|0.007|0.0|0.0|0.0|20.0|1.7| 0.0| 9.6| 56.0|101.0|157.0|2.0|0.0|114.0|118.0|118.0|10.0|-1.0|
|533|130.0|  0.0|  0.0|  0.0|0.0|0.0|0.0|53.0|0.6|51.0| 5.0| 19.0|114.0|133.0|2.0|0.0|120.0|122.0|122.0| 3.0|-1.0|
|534|122.0|  0.0|  0.0|0.005|0.0|0.0|0.0|37.0|0.8|11.0|10.0| 26.0|109.0|135.0|2.0|0.0|123.0|125.0|126.0| 2.0| 0.0|
|535|127.0|0.011|0.033|0.002|0.0|0.0|0.0|38.0|1.4| 0.0| 7.2|113.0| 61.0|174.0|6.0|1.0|129.0|144.0|141.0|45.0| 1.0|
+---+-----+-----+-----+-----+---+---+---+----+---+----+----+-----+-----+-----+--

In [45]:
# create feature table
from databricks.feature_engineering import FeatureEngineeringClient
FEATURE_TABLE_NAME = f"{TABLE_NAME}_feature"
FEATURE_TABLE_PATH = get_catalog_path(FEATURE_TABLE_NAME)
print(FEATURE_TABLE_PATH)

workspace.lilypond_experiments.t_cardio_feature


In [48]:
if DATASET_PERSIST_MODE:
	feClient = FeatureEngineeringClient()
	feClient.create_table(
		name=FEATURE_TABLE_PATH,
		primary_keys=primary_key,
		df=feature_dbdf,
		description=f"{DATASET_NAME} features (original)",
		tags={"source": "bronze", "format": "delta"}
	)

2026/09/19 13:26:30 INFO databricks.ml_features._compute_client._compute_client: Setting columns ['id'] of table 'workspace.lilypond_experiments.t_cardio_feature' to NOT NULL.
2026/09/19 13:26:32 INFO databricks.ml_features._compute_client._compute_client: Setting Primary Keys constraint ['id'] on table 'workspace.lilypond_experiments.t_cardio_feature'.
2026/09/19 13:26:54 INFO databricks.ml_features._compute_client._compute_client: Created feature table 'workspace.lilypond_experiments.t_cardio_feature'.


## Register representation model in Databricks

In [ ]:
MODEL_NAME = f"som-{DATASET_NAME.lower()}"
MODEL_PATH = get_catalog_path(MODEL_NAME)
EXPERIMENT_NAME = f"/Users/matebalogh@ophelia-rnd.dev/Bianor_LilypondExperiments_{DATASET_NAME}"
print(MODEL_PATH, EXPERIMENT_NAME)

In [ ]:
if MODEL_REGISTRATION_MODE:

	mlflow.set_experiment(experiment_name=EXPERIMENT_NAME)

	class MLflowSomModelWrapper(mlflow.pyfunc.PythonModel):
		from typing import Any

		def __init__(self, model:SomRepresentation, scaler):
			self.model = model
			self.scaler = scaler
		def predict(self, context, model_input, params: dict[str, Any] | None = None):
			"""First transforms the input data via scaler, then predicts the winner node of the SOM."""
			return [self.model.som.winner(x) for x in self.scaler.transform(model_input.to_numpy())]

	with mlflow.start_run():
		model = MLflowSomModelWrapper(som_rep, scaler)

		mlflow.log_metric("QE", som_rep.quantization_error)
		mlflow.log_metric("TE", som_rep.topographic_error)

		mlflow.pyfunc.log_model(
			python_model=model,
			name=MODEL_NAME,
			input_example=pd.DataFrame(Xs_hist_normal[:3]),
			pip_requirements=[
				"numpy",
				"pandas",
				"scikit-learn==1.5.2",
				"mlflow",
				"minisom",
			],
			registered_model_name=MODEL_PATH
		)

else: print("Skipping MLflow model registration.")

In [ ]:
version = 1

registered_model = f"{MODEL_NAME}/{version}"
print(registered_model)

registered_model_location = get_catalog_path(registered_model)
print(registered_model_location)

## Register metadata in Bianor

In [34]:
from bianor_databricks_kit import BianorRecordManager
bianor_recorder = BianorRecordManager(catalog=CATALOG, schema=SCHEMA, spark=spark)

In [ ]:
if MODEL_REGISTRATION_MODE:
	# first registration
	# bianor_recorder.new_representation(name=f"{DATASET_NAME} Representation", som_model_location=registered_model_location, features_location="c.s.t")
	pass
else: print("Skipping Bianor metadata registration.")

In [31]:
rep_id = "c6556bb8-4d71-4775-a1e4-86ecd87c5c11"

In [41]:
if MODEL_REGISTRATION_MODE:

	# register HISTORICAL projection layers

	class_names = {1: "Normal", 2: "Suspect", 3: "Pathologic"}
	sample_locations = [f"V_{DATASET_NAME}_class_{cn}".lower() for cn in class_names]
	colors = ["#FF6347", "#FF4500", "#FF00FF"]
	names = [f"Historical data - {cn}" for cn in class_names.values()]
	names[0] = f"Historical (train) data - {class_names[1]}"

	for color, name, sample_loc in zip(colors, names, sample_locations):
		marker_dict = dict(
			color=color,
			symbol="star-diamond",
			opacity=0.9
		)

		proj_id = bianor_recorder.new_projection_layer(
			name=name,
			marker_dict=marker_dict,
			samples_location=get_catalog_path(sample_loc)
		)

		bianor_recorder.new_map_representation_projection_layer(representation_id=rep_id, projection_layer_id=proj_id)

else: print("Skipping Bianor metadata registration.")

In [35]:
if MODEL_REGISTRATION_MODE:

	# register HOLDOUT ground-truth / prediction layers

	sample_location = f"V_{DATASET_NAME}_holdout_ground_truth".lower()
	name = "Holdout predictions (ground truth)"

	pred_id = bianor_recorder.new_prediction(
		name=name,
		prediction_location=get_catalog_path(sample_location)
	)

	bianor_recorder.new_map_representation_prediction(representation_id=rep_id, prediction_id=pred_id)

else: print("Skipping Bianor metadata registration.")